# Clase 8 — El framework: Strategy + Backtest

El corazón del curso: una estrategia solo reacciona al libro y devuelve acciones. El Backtest la cablea con el mercado y el portfolio. Cualquier estrategia se enchufa igual.

**Hoy construyes:** interfaz Strategy (ABC) y el runner Backtest.

## Cómo usar este cuaderno

- **Núcleo (en clase):** ejercicios 1 a 3.
- **Si vamos bien:** ejercicios 4 en adelante.
- **Casa / auxiliares:** el cuaderno `*_auxiliary.ipynb`.

Inténtalo, ejecuta la comprobación (`assert`) y mira la solución solo si te atascas.

## 1. Tu primera estrategia

**Practicas:** heredar de Strategy.

Define `BuyOnce(Strategy)`: en el primer libro envía una market buy de 0.5; después nada.

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType
class BuyOnce(Strategy):
    def __init__(self):
        self.done = False
    def on_book_update(self, book):
        pass

In [ ]:
from exchange import Market, Backtest
r = Backtest(Market.sample(), BuyOnce()).run()
assert r.n_fills >= 1 and r.final_position > 0
print('ok  fills=%d pos=%.2f' % (r.n_fills, r.final_position))

### Solución guiada

```python
class BuyOnce(Strategy):
    def __init__(self):
        self.done = False
    def on_book_update(self, book):
        if self.done:
            return []
        self.done = True
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.5, order_type=OrderType.MARKET))]
```

## 2. Corre el Backtest

**Practicas:** Backtest.run.

Corre `BuyOnce` con `Backtest(Market.sample(), BuyOnce()).run()`. Guarda `result` y `equity`.

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class BuyOnce(Strategy):
    def __init__(self): self.done=False
    def on_book_update(self, book):
        if self.done: return []
        self.done=True
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.5, order_type=OrderType.MARKET))]
result = None
equity = None

In [ ]:
assert result.n_steps == 500
assert isinstance(equity, float)
print('ok ', result)

### Solución guiada

```python
result = Backtest(Market.sample(), BuyOnce()).run()
equity = result.final_equity
```

## 3. Reacciona a tus fills

**Practicas:** el hook on_fill.

Define `CountingBuyer(Strategy)` que cuente sus fills en `self.n` vía `on_fill`. Compra 0.1 cada paso.

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class CountingBuyer(Strategy):
    def __init__(self):
        self.n = 0
    def on_book_update(self, book):
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET))]
    def on_fill(self, fill):
        pass

In [ ]:
s = CountingBuyer()
Backtest(Market.sample(), s).run()
assert s.n >= 1, 'on_fill debe haberse llamado'
print('ok  on_fill llamado %d veces' % s.n)

### Solución guiada

```python
class CountingBuyer(Strategy):
    def __init__(self):
        self.n = 0
    def on_book_update(self, book):
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET))]
    def on_fill(self, fill):
        self.n += 1
```

## 4. Polimorfismo: cambia la estrategia, no el runner

**Practicas:** intercambiar subclases.

Define `SellOnce` (igual que BuyOnce pero vende). Córrela en el MISMO Backtest. Guarda `pos` (debe ser < 0).

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class SellOnce(Strategy):
    def __init__(self): self.done=False
    def on_book_update(self, book):
        pass

pos = None

In [ ]:
assert pos < 0, 'SellOnce deja posición corta — mismo runner, otra estrategia'
print('ok  pos=%.2f' % pos)

### Solución guiada

```python
class SellOnce(Strategy):
    def __init__(self): self.done=False
    def on_book_update(self, book):
        if self.done: return []
        self.done=True
        return [NewOrder(Order('BTCUSDT', Side.SELL, 0.5, order_type=OrderType.MARKET))]
pos = Backtest(Market.sample(), SellOnce()).run().final_position
```

## Cierre

Escribe una subclase de Strategy y enchúfala al mismo Backtest. Eso es polimorfismo, y es lo que hace todo modular.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.